# ### # **1. Creación del entorno**

**1.1 Creamos nuetrs estructura dentro dentro de Unity Catalogo**

In [0]:
# Crear catálogo
spark.sql("CREATE CATALOG IF NOT EXISTS `sesion1`")
# Crear esquema dentro del catálogo
spark.sql("CREATE SCHEMA IF NOT EXISTS `sesion1`.data")
# Crear volumen dentro del esquema
spark.sql("CREATE VOLUME IF NOT EXISTS `sesion1`.data.landing")

1.2 incorporación de datos de un volumen

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/nutrients_csvfile.csv -o /volumen/sesion1/landing/nutrients_csvfile.csv

### 1.3 Crear on una tabla 

In [0]:
%sql
CREATE TABLE sesion1.data.tabla_simple (
  id INT,
  letra STRING,
  valor DOUBLE
);

insert into sesion1.data.tabla_simple values (1,'A',10.5),(2,'B',20.5),(3,'C',30.75)

In [0]:
%m

In [0]:
%sql
CREATE OR REPLACE VIEW sesion1.data.vw_taba_simple AS
SELECT id, letra
FROM sesion1.data.tabla_simple
WHERE valor > 15;


###1.5 crear una función

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS `sesion1`.security")

In [0]:
%sql
CREATE OR REPLACE FUNCTION sesion1.security.fn_mayor_que_10(x DOUBLE)
RETURNS BOOLEAN
RETURN x > 15;

In [0]:
%sql
select * from sesion1.data.tabla_simple where sesion1.security.fn_mayor_que_10(valor);

#### 1.6 creacion de dataframe

In [0]:
path_data_demo = "volumen/sesion1/landing/data_demo.csv"
df = spark.read.format("csv").option("header", "true").load(path_data_demo)
df.write.mode("overwrite").saveAsTable("sesion1.security.data_demo")

In [0]:
display(df)

#### actividad de clase

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/ventas_2025.csv -o /Volumes/sesion1/data/landing/ventas_2025.csv

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("id", StringType(), True),
    StructField("fecha", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio", DoubleType(), True)
])

csv_url = "/Volumes/sesion1/data/landing/ventas_2025.csv"
df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .schema(schema) \
    .load(csv_url)

In [0]:
display(df_csv)

display()

####1.7 guardar la dataframe  en una tabala delta

In [0]:
df_csv.write.format("delta").mode("overwrite").saveAsTable("sesion1.data.products")


####modificar las tablas

In [0]:
%sql
insert into sesion1.data.products(id, fecha, producto, cantidad,precio) values ("A001", "2025-11-25", "Torre E-138", 3, 120000.50)

####2.2.2 Update

In [0]:
%sql
update sesion1.data.products 
set cantidad = 5, precio = 119000
where id = "1";

In [0]:
%sql
select * from sesion1.data.products
where id = "1";

#### 2.2.3 Merge

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import Row

# Obtener la tabla Delta
delta_table = DeltaTable.forName(spark, "sesion1.data.products")

# Crear DataFrame con nuevos registros
new_data = [
    Row(id="A002", fecha="2025-12-01", producto="Monitor X-200", cantidad=2, precio=45000.0),
    Row(id="A001", fecha="2025-11-25", producto="Torre E-138", cantidad=4, precio=118000.0)
]
df_new = spark.createDataFrame(new_data)

# Realizar el MERGE
delta_table.alias("target").merge(
    df_new.alias("source"),
    "target.id = source.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

####2.2.3 Time travel 

In [0]:
%sql
describe history sesion1.data.products

In [0]:
%sql
select * from sesion1.data.products version as of 1;

In [0]:
%sql
create table sesion1.data.products_v3
AS select *from sesion1.data.products version as of 3;

## 3. introducción a PySpark

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS sesion1.sparkintro")
spark.sql("CREATE VOLUME IF NOT EXISTS sesion1.sparkintro.landing")

##Almacenar datos volumen

In [0]:
%sh
curl -L https://raw.githubusercontent.com/williamcampos-026/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/spark_intro.csv -o /Volumes/sesion1/sparkintro/landing/spark_intro.csv

%sh
curl -L https://raw.githubusercontent.com/williamcampos-026/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/dim_spark_intro.csv -o /Volumes/sesion1/sparkintro/landing/dim_spark_intro.csv

In [0]:
sales_df = (spark.read
           .option("header", True)
           .option("inferSchema", True)
           .csv("/Volumes/sesion1/sparkintro/landing/spark_intro.csv")
          )
          display (sales_df)

### 2.4 Select(): elegir columnas


In [0]:
sales_simpl_df = sales_df.select("orde_id","order_date","conuntry","units_sold")
display(sales_simpl_df)

##3.4 filter()/where():filtrar filas

In [0]:
from pyspark.sql import functions as F


In [0]:
#Filtrar ventas de España
sales_spain_df = sales_df.filter(F.col("country") == "spain")
display(sales_spain_df) 

#### 3.5 withColomn:crear y transformar columnas

In [0]:
sales_with_total_df = sales_df.withColumn("total_sales",F.col("units_sold") * F.col("unit_price"))

#### groupby().agg():agregaciones

In [0]:
sales_country_df = sales_df_goupby("conuntry").agg(F.sum("total_unit_sold", 
                                                         agg(F.sum("unit_sold".alias("total_sales_aount")))
                                                         display(sales_country_df)

#### 3.8 Join(): combiar datos de distintas tabalas

In [0]:
product_dim_df = 
spark.read
.option("header", True)
.option("inferScema", True)
.csv("/volumes/sesion1/")

In [0]:
sales_complete_df = (sales_with_total_df.alias ("s")
                     .join(product_dim_df.alias("p"),on))

In [0]:
sales_complete_df.write.format("delta").mode("overwrite").saveAsTable ("sesion1.sparkintro.sales_complete")